In [1]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
Python executable: c:\DATA_SCIENCE\PORTAFOLIO\VibeSig\.venv\Scripts\python.exe


In [2]:
from datasets import load_dataset
import pandas as pd

c:\DATA_SCIENCE\PORTAFOLIO\VibeSig\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("Yelp/yelp_review_full")

c:\DATA_SCIENCE\PORTAFOLIO\VibeSig\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MARE\.cache\huggingface\hub\datasets--Yelp--yelp_review_full. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\DATA_SCIENCE\PORTAFOLIO\VibeSig\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarn

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [5]:
dataset["train"][0]

{'label': 4,
 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank."}

In [6]:
dataset["train"].features["label"]

ClassLabel(names=['1 star', '2 star', '3 stars', '4 stars', '5 stars'])

In [7]:
from collections import Counter

Counter(dataset["train"]["label"])

Counter({4: 130000, 1: 130000, 3: 130000, 0: 130000, 2: 130000})

In [8]:
samples = []

for label in range(5):
    subset = dataset["train"].filter(lambda x: x["label"] == label)
    subset = subset.shuffle(seed=42).select(range(3000))
    samples.append(subset)

Filter: 100%|██████████| 650000/650000 [00:03<00:00, 205387.29 examples/s]


In [9]:
for label, sample in enumerate(samples):
    print(f"Label {label}: {len(sample):,} reviews")

Label 0: 3,000 reviews
Label 1: 3,000 reviews
Label 2: 3,000 reviews
Label 3: 3,000 reviews
Label 4: 3,000 reviews


In [10]:
from datasets import concatenate_datasets

sample_dataset = concatenate_datasets(samples)
sample_dataset = sample_dataset.shuffle(seed=42)

print(sample_dataset)

Dataset({
    features: ['label', 'text'],
    num_rows: 15000
})


In [11]:
def add_vibesig_labels(example):
    example["rating"] = example["label"] + 1

    if example["rating"] <= 2:
        example["actual_sentiment"] = "Negative"
    elif example["rating"] == 3:
        example["actual_sentiment"] = "Neutral"
    else:
        example["actual_sentiment"] = "Positive"

    return example


sample_dataset = sample_dataset.map(add_vibesig_labels)

print(sample_dataset)

Map: 100%|██████████| 15000/15000 [00:01<00:00, 14364.86 examples/s]

Dataset({
    features: ['label', 'text', 'rating', 'actual_sentiment'],
    num_rows: 15000
})


In [12]:
from collections import Counter

sentiment_counts = Counter(sample_dataset["actual_sentiment"])

print(sentiment_counts)

Counter({'Positive': 6000, 'Negative': 6000, 'Neutral': 3000})


In [13]:
for i in range(5):
    review = sample_dataset[i]

    print(f"Rating: {review['rating']} stars")
    print(f"Sentiment: {review['actual_sentiment']}")
    print(f"Review: {review['text'][:300]}")
    print("-" * 80)

Rating: 5 stars
Sentiment: Positive
Review: An incredibly classy restaurant. Went here to celebrate a birthday, and it was a top class experience. \n\nGo for the $50 suggested menu and add a drink. Then get ready for a delicious evening. Great ambiance, incredible service, and delicious food. My favorite restaurant of the year.\n\nCarmine is 
--------------------------------------------------------------------------------
Rating: 3 stars
Sentiment: Neutral
Review: This is a great place to come see both local and up and coming bands. I like that they always have different events held here such as Nickle Beer Nights, Karaoke Mondays, and Beer Garden+Food Truck Nights. Frist Friday's are always a blast!!! They open the back and play videos or movies on the wall.
--------------------------------------------------------------------------------
Rating: 1 stars
Sentiment: Negative
Review: Straight ass ghetto! A hookers paradise! Good place if you want to save money. Don't book cheap vacation 

In [14]:
df = sample_dataset.to_pandas()

df.head()

,label,text,rating,actual_sentiment
0,4,An incredibly classy restaurant. Went here to ...,5,Positive
1,2,This is a great place to come see both local a...,3,Neutral
2,0,Straight ass ghetto! A hookers paradise! Good ...,1,Negative
3,1,I don't get it here. This place is known all ...,2,Negative
4,0,Our server was fine and the food is just ok. ...,1,Negative


In [15]:
print("Shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Shape: (15000, 4)

Data types:
label               int64
text                  str
rating              int64
actual_sentiment      str
dtype: object

Missing values:
label               0
text                0
rating              0
actual_sentiment    0
dtype: int64


In [16]:
df.to_csv(
    "data/yelp_vibesig_sample.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.
